In [1]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')
COURSE_LIMIT_PATH ='/content/drive/MyDrive/MOOCCubeX/Dataset/course_limit_Nhom6.csv'
USER_PATH = '/content/drive/MyDrive/MOOCCubeX/Dataset/entities/user.json'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import json
import gzip
import bz2
import orjson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
def read_jsonl_in_batches(file_path, batch_size=10000, use_orjson=True):
    """
    Đọc file JSON Lines theo từng batch và yield DataFrame.

    Parameters
    ----------
    file_path : str
        Đường dẫn tới file JSONL (có thể nén .gz hoặc .bz2)
    batch_size : int
        Số dòng mỗi batch
    use_orjson : bool
        Sử dụng orjson để tăng tốc (nếu có cài đặt)

    Yields
    ------
    pd.DataFrame
        Batch dữ liệu dưới dạng DataFrame
    """

    # Chọn thư viện parse JSON
    json_loader = orjson.loads if use_orjson else json.loads

    # Chọn method mở file (hỗ trợ file nén)
    if file_path.endswith(".gz"):
        open_func = gzip.open
    elif file_path.endswith(".bz2"):
        open_func = bz2.open
    else:
        open_func = open

    total_lines = 0
    error_lines = 0
    batch = []

    try:
        with open_func(file_path, 'rt', encoding='utf-8') as file:
            for line_num, line in enumerate(file, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json_loader(line)
                    batch.append(obj)
                except Exception as e:
                    error_lines += 1
                    print(f"[Warning] Lỗi parse JSON tại dòng {line_num}: {e}")
                    continue

                if len(batch) >= batch_size:
                    yield pd.DataFrame(batch)
                    total_lines += len(batch)
                    batch = []

            # Batch cuối
            if batch:
                yield pd.DataFrame(batch)
                total_lines += len(batch)

    except FileNotFoundError:
        print(f"[Error] File không tồn tại: {file_path}")
    except Exception as e:
        print(f"[Error] Lỗi khi đọc file: {e}")

    print(f"[Done] Đọc xong {total_lines} dòng hợp lệ, {error_lines} dòng lỗi.")

In [3]:
import pandas as pd
course_limit_df = pd.read_csv(COURSE_LIMIT_PATH)
user_df = pd.concat(read_jsonl_in_batches(USER_PATH, batch_size=10000), ignore_index=True)
print(user_df.shape)
# Số lượng null của mỗi cột
print(user_df.isnull().sum())

[Done] Đọc xong 3330294 dòng hợp lệ, 0 dòng lỗi.


/tmp/ipython-input-1404054394.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  user_df = pd.concat(read_jsonl_in_batches(USER_PATH, batch_size=10000), ignore_index=True)


(3330294, 7)
id                     0
name                  54
gender                54
school                54
year_of_birth    3281764
course_order           0
enroll_time            0
dtype: int64


In [4]:
# Số lượng course_id duy nhất của  dataframe
print(course_limit_df['course_id'].nunique())

1092


In [5]:
# courses_not_in_user_data = course_limit_df[~course_limit_df['course_id'].isin(user_course_use_labling['course_id'])]

# Check for two or more nulls in 'assignment', 'video', 'exam' columns for these courses
num_nulls_in_relevant_cols = course_limit_df[['assignment', 'video', 'exam']].isnull().sum(axis=1)

# Calculate the sum of assignment, video, and exam, treating NaNs as 0
sum_of_relevant_cols = course_limit_df['assignment'].fillna(0) + course_limit_df['video'].fillna(0) + course_limit_df['exam'].fillna(0)

# Filter courses based on two conditions: less than 2 nulls in relevant columns AND sum of relevant columns equals 100
filtered_courses = course_limit_df[ (sum_of_relevant_cols != 100)]

print(f"Number of course_ids in course_limit_df not found in user_course_use_labling and with 2 or more nulls in assignment/video/exam: {filtered_courses['course_id'].nunique()}")
display(filtered_courses.head(16))

Number of course_ids in course_limit_df not found in user_course_use_labling and with 2 or more nulls in assignment/video/exam: 118


,course_id,name,field_x,num_field_x,prerequisites,num_prerequisites,about,resource,count_course_id,certificate,assignment,video,exam,type,start_date,end_date,season
25,C_680920,华为成功之道：人力资源管理的最佳实践,[],0,[],0,28年，从3个人到15万人，从2.1万元到2880亿的世界500强公司。华为，凭什么成功？凭...,"[{'titles': ['（一）华为的企业文化', '简单驱逐复杂', 'Video'],...",29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31,C_681001,国际旅游发展动态,['地理学'],1,['无'],1,This course covers‍ the following topics: Glob...,"[{'titles': ['TOURISM DEVELOPMENT: PAST, PRESE...",322,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38,C_681045,刑事审判心理学,"['法学', '心理学']",2,[],0,这门课将介绍行为科学如何改善刑事司法制度。,[{'titles': ['欢迎来到犯罪学101系列课程Welcome to CRIME10...,74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,C_681152,科学传播公开课,[],0,[],0,媒体朋友们，“一切新闻都是科技新闻”的时代已经来临，你是否愿意掌握科学传播的基础知识和技能？...,"[{'titles': ['第一章：科学传播重要吗？', '1.1科学传播重要吗？', '科...",30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,C_681411,逐梦心言——手语的学习,['外国语言文学'],1,[],0,《逐梦心言——手语的学习》在线开放课程基于《中国手语》的课程开发，是特殊教育专业的特色课程。...,"[{'titles': ['引言', '走近手语', '引言'], 'resource_id...",1511,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60,C_681696,英语政论,"['外国语言文学', '政治学']",2,[],0,《英语政论》是一门专业英语课，旨在通过批判性阅读国际事务领域学术文章提升英语综合运用能力。课...,"[{'titles': ['Course Introduction', 'A Brief I...",77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63,C_681719,泵与泵站,[],0,['水力学'],1,课程将从给排水工程中常用的水泵类型开始，以应用最普遍的离心泵为重点，分析水泵的基本构造与工作...,"[{'titles': ['0 绪论', '了解本课程', '了解本课程'], 'resou...",446,1.0,60.0,20.0,30.0,1.0,"['2020-06-15', '2020-09-01', '2020-08-20', '20...","['2020-08-28', '2020-12-31', '2021-07-25']","['summer', 'fall', 'spring']"
65,C_681745,信息素养-互联网+时代的学习与生活,['计算机科学与技术'],1,[],0,《信息素养-互联网+时代的学习与生活》包括“互联网+时代的学习生活”、“信息的获取与鉴别”、...,"[{'titles': ['第1章 大咖访谈', '1.1 大学生如何提升计算思维和信息素养...",607,NaN,NaN,NaN,NaN,NaN,"['2020-01-01', '2020-02-21', '2020-09-01', '20...","['2020-02-20', '2020-07-31', '2020-12-31', '20...","['winter', 'spring', 'fall', 'spring']"
66,C_681760,大学生心理成长之路,['心理学'],1,[],0,"专题一 我是谁,专题二 人格使你与众不同,专题三 情绪管理与调控——情绪和谐的奥秘,专题四 ...","[{'titles': ['专题一\t我是谁', '第一节：自我意识的概念及结构', '自我...",807,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90,C_682516,国防教育——热点问题,['军队政治工作学'],1,[],0,本课程重点讲解我国国防面临的内外环境与威胁，探讨全球化进程中我国面临的各类国防热点问题，通过...,"[{'titles': ['第一章 习近平强军思想', '第一节 习近平强军思想产生的现实背...",196,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
course_limit_df.isna().sum()

,0
course_id,0
name,1
field_x,0
num_field_x,0
prerequisites,0
num_prerequisites,0
about,2
resource,0
count_course_id,0
certificate,88


In [7]:
# Số lượng course trong course_limit_df có tổng 'assignment', 'video', 'exam' = 100 và có start_date và end_date
course_limit_df[['assignment','video','exam']].fillna(0)
course_limit_df = course_limit_df[(course_limit_df['assignment'] + course_limit_df['video'] + course_limit_df['exam']) == 100]

In [8]:
# Lay nhung dong du lieu có start_date và end_date
course_limit_df = course_limit_df[(course_limit_df['start_date'].notna()) & (course_limit_df['end_date'].notna())]
course_limit_df.shape

(956, 17)

In [9]:
import ast
import pandas as pd
import numpy as np
import re

def parse_list_string(s):
    """
    Chuyển chuỗi biểu diễn list (đặc biệt list ngày tháng) sang mảng Python an toàn.
    Tự động xử lý:
    - Thiếu ngoặc vuông [ ]
    - Thiếu dấu nháy giữa các phần tử
    - Ngày không có dấu nháy
    - Thiếu dấu nháy đóng ở phần tử cuối
    - Chuỗi None, nan, rỗng
    """
    # 🔹 Bước 1: Kiểm tra kiểu dữ liệu
    if not isinstance(s, str) or s.strip().lower() in ["", "none", "nan"]:
        return []

    s = s.strip()
    # FIX: Remove trailing semicolon that causes SyntaxError
    if s.endswith(';'):
        s = s[:-1].strip()

    # 🔹 Bước 2: Đảm bảo có ngoặc vuông
    if not s.startswith("["):
        s = "[" + s
    if not s.endswith("]"):
        s = s + "]"

    # 🔹 Bước 3: Thêm dấu nháy quanh các ngày (nếu bị thiếu)
    # Ví dụ: [2020-01-01, 2020-07-31] -> ['2020-01-01', '2020-07-31']
    s = re.sub(r"(?<=\[)\s*([0-9]{4}-[0-9]{2}-[0-9]{2})", r"'\1'", s)
    s = re.sub(r",\s*([0-9]{4}-[0-9]{2}-[0-9]{2})", r", '\1'", s)

    # 🔹 Bước 4: Vá lỗi thiếu dấu nháy bị đứt giữa chừng
    # Ví dụ: '2020-07-31, '2020-12-31' -> '2020-07-31', '2020-12-31'
    s = re.sub(r"(\d{4}-\d{2}-\d{2}),\s+'", r"\1', '", s)

    # 🔹 Bước 5: Vá lỗi thiếu dấu nháy đóng ở phần tử cuối
    # Ví dụ: ['2020-01-10', '2020-07-10', '2020-12-31', '2021-07-25]
    s = re.sub(r"([0-9]{4}-[0-9]{2}-[0-9]{2})\]", r"'\1']", s)

    # 🔹 Bước 6: Loại bỏ dấu nháy thừa hoặc lỗi format
    s = re.sub(r"''+", "'", s)

    # 🔹 Bước 7: Parse chuỗi an toàn
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return []

# Test the user's specific string to show it can be parsed
user_test_string = "['2019-09-27', '2020-02-17', '2020-09-01', '2021-02-28']"
parsed_user_string = parse_list_string(user_test_string)
print(f"User's string parsed: {parsed_user_string}, Type: {type(parsed_user_string)}")

# Parse columns
for col in ['start_date', 'end_date']:
    course_limit_df[f'{col}_list'] = course_limit_df[col].apply(parse_list_string)
    course_limit_df[f'{col}_list_len'] = course_limit_df[f'{col}_list'].apply(len)

# Filter
course_limit_filtered_len_df = course_limit_df[
    course_limit_df['start_date_list_len'] == course_limit_df['end_date_list_len']
].copy()

print("Before:", course_limit_df.shape)
print("After:", course_limit_filtered_len_df.shape)

# Explode
course_limit_exploded_df = course_limit_filtered_len_df.explode(['start_date_list', 'end_date_list'], ignore_index=True)

User's string parsed: ['2019-09-27', '2020-02-17', '2020-09-01', '2021-02-28'], Type: <class 'list'>
Before: (956, 21)
After: (908, 21)


In [10]:
# Ensure date columns in course_limit_exploded_df are in datetime format
course_limit_exploded_df['start_date_list'] = pd.to_datetime(course_limit_exploded_df['start_date_list'], errors='coerce')
course_limit_exploded_df['end_date_list'] = pd.to_datetime(course_limit_exploded_df['end_date_list'], errors='coerce')

# Calculate the duration of the course in days
course_limit_exploded_df['course_duration'] = (course_limit_exploded_df['end_date_list'] - course_limit_exploded_df['start_date_list']).dt.days

# Display the head of the DataFrame with the new column
course_limit_exploded_df[['course_duration']].describe()

,course_duration
count,2439.000000
mean,161.780238
std,88.020921
min,-4.000000
25%,117.000000
50%,131.000000
75%,202.000000
max,506.000000


In [11]:
# Khóa học có start_date_list < end_date_list

course_limit_exploded_df = course_limit_exploded_df[course_limit_exploded_df['start_date_list'] < course_limit_exploded_df['end_date_list']]

In [12]:
course_limit_exploded_df[['course_duration']].describe()

,course_duration
count,2437.000000
mean,161.914649
std,87.931788
min,18.000000
25%,117.000000
50%,131.000000
75%,202.000000
max,506.000000


In [13]:
course_limit_exploded_df.columns

Index(['course_id', 'name', 'field_x', 'num_field_x', 'prerequisites',
       'num_prerequisites', 'about', 'resource', 'count_course_id',
       'certificate', 'assignment', 'video', 'exam', 'type', 'start_date',
       'end_date', 'season', 'start_date_list', 'start_date_list_len',
       'end_date_list', 'end_date_list_len', 'course_duration'],
      dtype='object')

In [14]:
# Loại bỏ những dòng dữ liệu không có thông tin bắt đầu và kết thúc
course_limit_exploded_df = course_limit_exploded_df[(course_limit_exploded_df['start_date_list'].notna()) & course_limit_exploded_df['end_date_list'].notna()]
course_limit_exploded_df.shape

(2437, 22)

In [15]:
# Create a new DataFrame with selected columns and rename them
course_dates_df = course_limit_exploded_df[['course_id', 'start_date_list', 'end_date_list','assignment', 'video', 'exam','course_duration']].rename(columns={'start_date_list': 'start_date', 'end_date_list': 'end_date'})

# Display the head of the new DataFrame
display(course_dates_df.head())

,course_id,start_date,end_date,assignment,video,exam,course_duration
0,C_674968,2019-10-28,2020-01-31,50.0,10.0,40.0,95.0
1,C_674968,2020-01-06,2020-06-30,50.0,10.0,40.0,176.0
2,C_674968,2020-08-06,2020-12-31,50.0,10.0,40.0,147.0
3,C_674968,2021-01-04,2021-07-25,50.0,10.0,40.0,202.0
4,C_674971,2019-11-04,2020-01-12,60.0,40.0,0.0,69.0


In [16]:
user_course_df = user_df[['id', 'course_order','enroll_time']].rename(columns={'id': 'user_id'}).explode(['course_order','enroll_time'], ignore_index=True)

In [17]:
# Đổi tên cột course_order thành course_id và thêm C_ vào trước mỗi dòng của course_id
user_course_df = user_course_df.rename(columns={'course_order': 'course_id'}).assign(course_id=lambda x: 'C_' + x['course_id'].astype(str))
user_course_df

,user_id,course_id,enroll_time
0,U_22,C_682129,2019-10-12 10:28:02
1,U_22,C_2294668,2020-11-21 14:03:28
2,U_24,C_597214,2019-05-20 16:06:48
3,U_24,C_605512,2019-05-24 19:34:43
4,U_24,C_597211,2019-06-11 02:50:04
...,...,...,...
11807085,U_34712115,C_770738,2020-10-12 07:17:41
11807086,U_34712115,C_676937,2020-10-20 20:32:10
11807087,U_34712115,C_694136,2020-10-20 20:33:38
11807088,U_34712115,C_1738993,2020-10-20 20:35:18


In [18]:
# Merge course_dates_df voi user_course_df
course_dates_labling_df = pd.merge(course_dates_df, user_course_df, on='course_id')

In [19]:
course_dates_labling_df

,course_id,start_date,end_date,assignment,video,exam,course_duration,user_id,enroll_time
0,C_674968,2019-10-28,2020-01-31,50.0,10.0,40.0,95.0,U_11731,2019-12-24 15:21:23
1,C_674968,2019-10-28,2020-01-31,50.0,10.0,40.0,95.0,U_13771,2019-12-21 22:36:42
2,C_674968,2019-10-28,2020-01-31,50.0,10.0,40.0,95.0,U_21726,2020-08-22 20:44:28
3,C_674968,2019-10-28,2020-01-31,50.0,10.0,40.0,95.0,U_47328,2020-04-18 16:29:49
4,C_674968,2019-10-28,2020-01-31,50.0,10.0,40.0,95.0,U_91340,2020-02-16 12:53:42
...,...,...,...,...,...,...,...,...,...
9961984,C_2343056,2021-03-01,2021-07-25,60.0,40.0,0.0,146.0,U_15312993,2020-12-05 06:10:47
9961985,C_2343056,2021-03-01,2021-07-25,60.0,40.0,0.0,146.0,U_15402003,2020-12-05 08:58:11
9961986,C_2343056,2021-03-01,2021-07-25,60.0,40.0,0.0,146.0,U_29032397,2020-12-07 13:06:23
9961987,C_2343056,2021-03-01,2021-07-25,60.0,40.0,0.0,146.0,U_29664657,2020-12-04 11:33:38


In [25]:
# Create an explicit copy to avoid SettingWithCopyWarning
course_dates_labling_df = course_dates_labling_df.copy()

# Convert 'enroll_time', 'start_date', and 'end_date' to datetime objects
course_dates_labling_df['enroll_time'] = pd.to_datetime(course_dates_labling_df['enroll_time'], errors='coerce')
course_dates_labling_df['start_date'] = pd.to_datetime(course_dates_labling_df['start_date'], errors='coerce')
course_dates_labling_df['end_date'] = pd.to_datetime(course_dates_labling_df['end_date'], errors='coerce')


# Lọc những dòng dữ liệu có enroll_time sau ngày start_date và trước hoặc đúng ngày kết thúc
course_dates_labling_df = course_dates_labling_df[(course_dates_labling_df['enroll_time'] >= course_dates_labling_df['start_date']) & (course_dates_labling_df['enroll_time'] < course_dates_labling_df['end_date'])].copy()

# Display the shape of the filtered DataFrame
print(course_dates_labling_df.shape)

(2445594, 10)


In [26]:
course_dates_labling_df

,course_id,start_date,end_date,assignment,video,exam,course_duration,user_id,enroll_time,enrollment_to_end
0,C_674968,2019-10-28,2020-01-31,50.0,10.0,40.0,95.0,U_11731,2019-12-24 15:21:23,37
1,C_674968,2019-10-28,2020-01-31,50.0,10.0,40.0,95.0,U_13771,2019-12-21 22:36:42,40
7,C_674968,2019-10-28,2020-01-31,50.0,10.0,40.0,95.0,U_149379,2020-01-06 11:21:46,24
14,C_674968,2019-10-28,2020-01-31,50.0,10.0,40.0,95.0,U_364446,2019-12-24 23:43:18,37
16,C_674968,2019-10-28,2020-01-31,50.0,10.0,40.0,95.0,U_678640,2019-12-10 15:23:06,51
...,...,...,...,...,...,...,...,...,...,...
9961894,C_2343056,2020-12-04,2021-02-28,60.0,40.0,0.0,86.0,U_15312993,2020-12-05 06:10:47,84
9961895,C_2343056,2020-12-04,2021-02-28,60.0,40.0,0.0,86.0,U_15402003,2020-12-05 08:58:11,84
9961896,C_2343056,2020-12-04,2021-02-28,60.0,40.0,0.0,86.0,U_29032397,2020-12-07 13:06:23,82
9961897,C_2343056,2020-12-04,2021-02-28,60.0,40.0,0.0,86.0,U_29664657,2020-12-04 11:33:38,85


In [27]:
# Calculate the time from enrollment to the end of the course
course_dates_labling_df['enrollment_to_end'] = (course_dates_labling_df['end_date'] - course_dates_labling_df['enroll_time']).dt.days

# Sort by user_id, course_id, and enrollment_to_end (descending) to keep the max 'enrollment_to_end' after dropping duplicates
course_dates_labling_df_sorted = course_dates_labling_df.sort_values(by=['user_id', 'course_id', 'enrollment_to_end'], ascending=[True, True, False])

# Drop duplicates based on user_id and course_id, keeping the one with the maximum enrollment_to_end
course_dates_labling_reduced_df = course_dates_labling_df_sorted.drop_duplicates(subset=['user_id', 'course_id'], keep='first').copy()

# Calculate the duration of the course (re-calculating for consistency, though it might be present in upstream steps)
course_dates_labling_reduced_df['course_duration'] = (course_dates_labling_reduced_df['end_date'] - course_dates_labling_reduced_df['start_date']).dt.days

# Display the shape and head of the reduced DataFrame
print("Shape of the reduced DataFrame:")
print(course_dates_labling_reduced_df.shape)
display(course_dates_labling_reduced_df.head())

Shape of the reduced DataFrame:
(2399533, 10)


,course_id,start_date,end_date,assignment,video,exam,course_duration,user_id,enroll_time,enrollment_to_end
9038731,C_2033958,2020-09-04,2020-12-31,40.0,30.0,30.0,118,U_10000,2020-10-27 09:07:30,64
8996923,C_1925903,2020-08-12,2020-12-31,30.0,30.0,40.0,141,U_1000129,2020-09-06 14:37:17,115
9030610,C_1992970,2020-08-27,2020-12-31,30.0,30.0,40.0,126,U_1000129,2020-09-06 14:45:37,115
526317,C_680884,2020-09-01,2020-12-31,80.0,20.0,0.0,121,U_1000129,2020-10-16 10:41:09,75
9435974,C_697791,2020-01-20,2020-07-31,80.0,20.0,0.0,193,U_1000342,2020-02-28 10:27:22,153


In [28]:
# Display the head of the DataFrame with the new columns
course_dates_labling_reduced_df[['course_duration','enrollment_to_end']].describe()

,course_duration,enrollment_to_end
count,2.399533e+06,2.399533e+06
mean,1.353614e+02,8.056630e+01
std,5.510461e+01,5.703925e+01
min,1.800000e+01,0.000000e+00
25%,1.000000e+02,4.200000e+01
50%,1.210000e+02,7.200000e+01
75%,1.600000e+02,1.080000e+02
max,5.060000e+02,5.050000e+02


In [30]:
# Những dòng dữ liệu  trong course_dates_labling_reduced_df có enrollment_to_end <7
course_dates_labling_reduced_df = course_dates_labling_reduced_df[course_dates_labling_reduced_df['enrollment_to_end'] >= 7]

In [31]:
course_dates_labling_reduced_df

,course_id,start_date,end_date,assignment,video,exam,course_duration,user_id,enroll_time,enrollment_to_end
9038731,C_2033958,2020-09-04,2020-12-31,40.0,30.0,30.0,118,U_10000,2020-10-27 09:07:30,64
8996923,C_1925903,2020-08-12,2020-12-31,30.0,30.0,40.0,141,U_1000129,2020-09-06 14:37:17,115
9030610,C_1992970,2020-08-27,2020-12-31,30.0,30.0,40.0,126,U_1000129,2020-09-06 14:45:37,115
526317,C_680884,2020-09-01,2020-12-31,80.0,20.0,0.0,121,U_1000129,2020-10-16 10:41:09,75
9435974,C_697791,2020-01-20,2020-07-31,80.0,20.0,0.0,193,U_1000342,2020-02-28 10:27:22,153
...,...,...,...,...,...,...,...,...,...,...
7446390,C_1774965,2020-04-27,2021-08-31,50.0,20.0,30.0,491,U_999821,2020-05-02 17:54:09,485
1361737,C_696724,2019-11-18,2020-02-29,80.0,20.0,0.0,103,U_999821,2019-12-30 06:22:37,60
1480518,C_696827,2019-12-25,2020-03-31,70.0,30.0,0.0,97,U_999856,2020-02-25 16:13:15,34
8704178,C_947252,2020-02-17,2020-07-31,55.0,10.0,35.0,165,U_999856,2020-03-01 16:09:18,151


In [34]:
course_dates_labling_reduced_df[['course_duration','enrollment_to_end']].describe()

,course_duration,enrollment_to_end
count,2.324544e+06,2.324544e+06
mean,1.360507e+02,8.306140e+01
std,5.536847e+01,5.620578e+01
min,1.800000e+01,7.000000e+00
25%,1.010000e+02,4.500000e+01
50%,1.210000e+02,7.400000e+01
75%,1.600000e+02,1.100000e+02
max,5.060000e+02,5.050000e+02


In [33]:
# Luu tru dataframe
course_dates_labling_reduced_df.to_csv('/content/drive/MyDrive/MOOCCubeX/Dataset/Data_cleaned/user_reduce_use_labeling.csv')

In [37]:
course_dates_labling_reduced_df['25%_enrollment_to_end'] = 0.25 * course_dates_labling_reduced_df['enrollment_to_end']
course_dates_labling_reduced_df['50%_enrollment_to_end'] = 0.50 * course_dates_labling_reduced_df['enrollment_to_end']
course_dates_labling_reduced_df['75%_enrollment_to_end'] = 0.75 * course_dates_labling_reduced_df['enrollment_to_end']
course_dates_labling_reduced_df['90%_enrollment_to_end'] = 0.90 * course_dates_labling_reduced_df['enrollment_to_end']

course_dates_labling_reduced_df[['course_duration','enrollment_to_end','25%_enrollment_to_end','50%_enrollment_to_end','75%_enrollment_to_end','90%_enrollment_to_end']].describe()

,course_duration,enrollment_to_end,25%_enrollment_to_end,50%_enrollment_to_end,75%_enrollment_to_end,90%_enrollment_to_end
count,2.324544e+06,2.324544e+06,2.324544e+06,2.324544e+06,2.324544e+06,2.324544e+06
mean,1.360507e+02,8.306140e+01,2.076535e+01,4.153070e+01,6.229605e+01,7.475526e+01
std,5.536847e+01,5.620578e+01,1.405145e+01,2.810289e+01,4.215434e+01,5.058520e+01
min,1.800000e+01,7.000000e+00,1.750000e+00,3.500000e+00,5.250000e+00,6.300000e+00
25%,1.010000e+02,4.500000e+01,1.125000e+01,2.250000e+01,3.375000e+01,4.050000e+01
50%,1.210000e+02,7.400000e+01,1.850000e+01,3.700000e+01,5.550000e+01,6.660000e+01
75%,1.600000e+02,1.100000e+02,2.750000e+01,5.500000e+01,8.250000e+01,9.900000e+01
max,5.060000e+02,5.050000e+02,1.262500e+02,2.525000e+02,3.787500e+02,4.545000e+02
